In [ ]:
# idk havent found a better way sorry
import os, sys
sys.path.insert(0, os.path.abspath("../.."))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from src import fdm_schemes
from src.iter_schemes import jacobi, gauss_seidel, sor
from src.grid import rectangle_sink, combine_sinks, make_grid, empty_insulator, rectangle_insulator

In [ ]:
# Choose a base font size
base_fs = 14

# Global update (applies for the rest of this Python session)
mpl.rcParams.update({
    'font.size': base_fs,                   # default text size
    'axes.titlesize': base_fs * 1.1,       # axes title
    'axes.labelsize': base_fs,              # x/y labels
    'xtick.labelsize': base_fs * 0.9,       # x tick labels
    'ytick.labelsize': base_fs * 0.9,       # y tick labels
    'legend.fontsize': base_fs * 0.9,       # legend text
    'legend.title_fontsize': base_fs * 0.9, # legend title
    'figure.titlesize': base_fs * 1.3,      # figure suptitle
    'figure.figsize': (8, 4.5),             # optional default figure size
})

## 1.6 

### H

In [ ]:
N= 50

cJ, dJ = jacobi(N)
cG, dG = gauss_seidel(N)
cS, dS, _, _ = sor(N, omega=1.8)   


In [ ]:
plt.imshow(cJ, aspect='auto')
plt.colorbar()
plt.show()


In [ ]:
y = np.linspace(1, 0, N)      
analytic = y                     

profJ = cJ.mean(axis=1)
profG = cG.mean(axis=1)
profS = cS.mean(axis=1)

errJ = np.max(np.abs(profJ - analytic))
errG = np.max(np.abs(profG - analytic))
errS = np.max(np.abs(profS - analytic))



In [ ]:

def calculate_and_plot_mse(c, N, analytic):
    """Calculate the point-wise mean squared error (MSE) between the numerical solution and the analytic solution.
    Parameters:
    c (ndarray): The numerical solution array of shape (N, N).
    N (int): The number of grid points in each dimension.
    analytic (ndarray): The analytic solution array of shape (N,). """

    mse_errors = []
    
    for i in range(N):
        mse = np.mean((c[i, :] - analytic[i]) ** 2)
        mse_errors.append(mse)
    max_mse = max(mse_errors)
    return mse_errors, max_mse


mseJ, maxJ = calculate_and_plot_mse(cJ, N, analytic)
mseG, maxG = calculate_and_plot_mse(cG, N, analytic)
mseS, maxS = calculate_and_plot_mse(cS, N, analytic)

plt.figure(figsize=(10, 6))
plt.plot(mseJ, marker='o', label="Jacobi")
plt.plot(mseG, marker='o', label="Gauss-Seidel")
plt.plot(mseS, marker='o', label="SOR, ω=1.8")
plt.text(4, 0.9*maxJ, f'Max MSE Jacobi: {maxJ:.2e}', fontsize=12, color='tab:blue')
plt.text(17, 2.9*10**(-5), f'Max MSE Gauss-Seidel: {maxG:.2e}', fontsize=12, color='tab:orange')
plt.text(18, 0.8*10**(-5), f'Max MSE SOR: {maxS:.2e}', fontsize=12, color='tab:green')

plt.xlabel("x", fontsize=12)
plt.ylabel("Mean Squared Error", fontsize=12)
plt.title("Point-wise MSE", fontsize=15.5)
plt.grid()
plt.legend()
plt.show()

### I

In [ ]:
_, dj_2 = jacobi(N)
_, dg_2 = gauss_seidel(N)

omegas = [1.2, 1.5, 1.8, 1.9]
ds_2 = {}
for o in omegas:
    _, ds_2[o], _, _ = sor(N, omega=o)


In [ ]:
plt.figure(figsize=(7,5))

plt.semilogy(dj_2, label="Jacobi", alpha=0.9)
plt.semilogy(dg_2, label="Gauss–Seidel", alpha=0.9)

for o in omegas:
    plt.semilogy(ds_2[o], "--", label=f"SOR ω={o}", alpha=0.5)


plt.xlabel("k")
plt.ylabel(r"$\delta(k)$")
plt.title("Convergence Comparison for Iterative Schemes")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()


### J

In [ ]:

Ns = [20, 30, 50, 80, 100] 
tol = 1e-5
max_iter = 50000

o_opts = []
k_opts = []

for N in Ns:
    omegas = np.arange(1.0, 2.0, 0.05)
    ks = []
    for o in omegas:
        _, _, k, conv = sor(N, omega=o, tol=tol, max_iter=max_iter)
        ks.append(k if conv else np.inf)
    ks = np.array(ks)

    w0 = omegas[np.argmin(ks)]

    omegas_finer = np.arange(max(1.0, w0-0.08), min(1.99, w0+0.08)+1e-12, 0.01)
    ks_finer = []
    for o in omegas_finer:
        _, _, k, conv = sor(N, omega=o, tol=tol, max_iter=max_iter)
        ks_finer.append(k if conv else np.inf)
    ks_finer = np.array(ks_finer)

    o_opt = omegas_finer[np.argmin(ks_finer)]
    k_opt = ks_finer.min()

    o_opts.append(o_opt)
    k_opts.append(k_opt)

   



In [ ]:
plt.figure()
plt.plot(Ns, o_opts, marker="o")
plt.xlabel("Grid size N")
plt.ylabel("Optimal ω")
plt.title("Optimal SOR relaxation parameter vs grid size")
plt.grid(True, alpha=0.3)
plt.show()

## K)

## placement impact

In [ ]:
def sink_area(sink):
    """Calculate the total area of the sink in the grid.
    Parameters:
    sink (ndarray): A 2D array representing the sink, where non-zero values indicate the presence of the sink."""
    
    return int(np.sum(sink))


In [ ]:
def square_sink(N, i0, j0, L):
    """Create a square sink in an N x N grid.
    Parameters:
    N (int): The size of the grid (N x N).
    i0 (int): The starting row index of the square sink.
    j0 (int): The starting column index of the square sink.
    L (int): The length of the sides of the square sink."""
    
    return rectangle_sink(N, i0, i0+L, j0, j0+L)


In [ ]:
N = 50
w = 1.9
tol = 1e-5

# same shape: 10x6 = 60 cells
sink_top = rectangle_sink(N, 1, 7, 2,  10)   
sink_mid = rectangle_sink(N, 22, 28, 0, 30)
sink_bot = rectangle_sink(N, 0, 0, 0, 0)   

# for name, s in [("top", sink_top), ("mid", sink_mid), ("bottom", sink_bot)]:
#     c, d, k = sor(N, omega=w, tol=tol, sink=s)
#     print(name, "area", sink_area(s), "iters", k)


c_s, d, k = sor(N, omega=w, tol=tol, sink=sink_top)
print(k)

In [ ]:

def placement_sweep(N, omega, L=8, step=5, tol=1e-5, max_iter=10000):
    """
    Sweep an LxL sink across interior placements in steps of 'step'.
    Parameters:
    N (int): The size of the grid (N x N).
    omega (float): The relaxation parameter for the SOR method.
    L (int): The length of the sides of the square sink.
    step (int): The step size for moving the sink across the grid.
    tol (float): The convergence tolerance for the SOR method.
    max_iter (int): The maximum number of iterations for the SOR method.
    """
    
    # valid i range: 1..N-2-L+1 so the square stays off top/bottom boundaries
    i_starts = np.arange(1, (N-1) - L + 1, step)
    j_starts = np.arange(0, N - L + 1, step)  # keep inside range (no wrap)

    K = np.empty((len(i_starts), len(j_starts)), dtype=float)

    for a, i0 in enumerate(i_starts):
        for b, j0 in enumerate(j_starts):
            s = square_sink(N, i0, j0, L)
            _, _, k, _ = sor(N, omega=omega, tol=tol, max_iter=max_iter, sink=s)
            K[a, b] = k if k < max_iter else np.inf

    return i_starts, j_starts, K

In [ ]:
N = 50
tol = 1e-5
max_iter = 10000
omega = 1.92   

i_starts, j_starts, K = placement_sweep(N, omega, L=6, step=3, tol=tol, max_iter=max_iter)

print("min k:", np.nanmin(K), "max k:", np.nanmax(K))

In [ ]:
plt.figure()
# plt.imshow(K, aspect="auto")
plt.imshow(
    K,
    origin="upper",
    aspect="auto",
    extent=[j_starts[0], j_starts[-1], i_starts[-1], i_starts[0]]
)
plt.colorbar(label="iterations k")
plt.xticks(ticks=np.arange(len(j_starts)), labels=j_starts)
plt.yticks(ticks=np.arange(len(i_starts)), labels=i_starts)
plt.xlabel("j0 (square left edge)")
plt.ylabel("i0 (square top edge)")
plt.title(f"Iteration count vs sink placement (L=6, step=3, ω={omega})")
plt.show()

## area checks

In [ ]:
def random_square_sink(N, L, rng):
    """Generate a random square sink placement in an N x N grid.
    Parameters:
    N (int): The size of the grid (N x N).
    L (int): The length of the sides of the square sink.
    rng (Generator): A NumPy random number generator instance."""

    i0 = rng.integers(1, (N-1) - L + 1)
    j0 = rng.integers(0, N - L + 1)
    return rectangle_sink(N, i0, i0+L, j0, j0+L)

In [ ]:
def area_sweep_squares(N, omega, L_values, n_trials=10, tol=1e-5, max_iter=10000, seed=0):
    """For each L in L_values, perform n_trials with random square sink placements and compute statistics on k.
    Parameters:
    N (int): The size of the grid (N x N).
    omega (float): The relaxation parameter for the SOR method.
    L_values (list of int): A list of side lengths for the square sinks to test.
    n_trials (int): The number of random placements to test for each L.
    tol (float): The convergence tolerance for the SOR method.
    max_iter (int): The maximum number of iterations for the SOR method.
    seed (int): The random seed for reproducibility."""
    
    rng = np.random.default_rng(seed)

    areas = []
    k_mean = []
    k_std = []
    k_ci95 = []
    k_all = {}  

    for L in L_values:
        ks = []
        for _ in range(n_trials):
            sink = random_square_sink(N, L, rng)
            _, _, k, conv = sor(N, omega=omega, tol=tol, max_iter=max_iter, sink=sink)
            ks.append(k if conv else np.inf)

        ks = np.array(ks, dtype=float)
        m = np.nanmean(ks)
        s = np.nanstd(ks, ddof=1)
        n = np.sum(~np.isnan(ks))
        ci95 = 1.96 * (s / np.sqrt(n))
        k_ci95.append(ci95)
        areas.append(L*L)
        k_mean.append(np.nanmean(ks))
        k_std.append(np.nanstd(ks))
        k_all[L] = ks

    return np.array(areas), np.array(k_mean), np.array(k_std), np.array(k_ci95), k_all

In [ ]:
N = 50
omega = 1.92
tol = 1e-5
max_iter = 10000

L_values = np.arange(1,  46, 1)  # to avoid errors from  i0 = rng.integers(1, (N-1) - L + 1)
areas, mean_k, std_k, ci95_k, k_all = area_sweep_squares(
    N, omega, L_values, n_trials=50, tol=tol, max_iter=max_iter, seed=1
)



In [ ]:
plt.figure()
plt.errorbar(areas, mean_k, yerr=ci95_k, fmt="o-")
plt.xlabel("Square sink area ")
plt.ylabel("Iterations to converge k (mean ± 95% CI)")
plt.title(f"Effect of sink area on convergence (N={N}, ω={omega}), trials = 50")
plt.grid(True, alpha=0.3)
plt.show()

## finding optimal omega

In [ ]:
def omega_opt_for_case(N, sink=None, tol=1e-5, max_iter=10000,
                       omega_min=1.0, omega_max=1.95, domega=0.05,
                       refine_halfwidth=0.08, refine_step=0.01):
    '''Find the optimal relaxation parameter ω for SOR for a given grid size N and sink configuration.
    Parameters:
    N (int): The size of the grid (N x N).
    sink (ndarray): A 2D array representing the sink configuration. If None, no sink is used.
    tol (float): The convergence tolerance for the SOR method.
    max_iter (int): The maximum number of iterations for the SOR method.
    omega_min (float): The minimum ω value to test in the initial coarse sweep.
    omega_max (float): The maximum ω value to test in the initial coarse sweep.
    domega (float): The step size for ω in the initial coarse sweep.
    refine_halfwidth (float): The half-width around the initial optimal ω to test in the refined sweep.
    refine_step (float): The step size for ω in the refined sweep.'''


    omegas = np.arange(omega_min, omega_max + 1e-12, domega)
    ks = []

    for w in omegas:
        _, _, k, conv = sor(N, omega=w, tol=tol, max_iter=max_iter, sink=sink)
        ks.append(k if conv else np.inf)

    ks = np.array(ks)
    w0 = omegas[np.argmin(ks)]

    omegas_f = np.arange(max(omega_min, w0-refine_halfwidth),
                         min(1.99, w0+refine_halfwidth) + 1e-12,
                         refine_step)

    ks_f = []
    for w in omegas_f:
        _, _, k, conv = sor(N, omega=w, tol=tol, max_iter=max_iter, sink=sink)
        ks_f.append(k if conv else np.inf)

    ks_f = np.array(ks_f)
    w_opt = omegas_f[np.argmin(ks_f)]
    k_opt = ks_f.min()

    return w_opt, k_opt, (omegas, ks), (omegas_f, ks_f)

In [ ]:
N = 50

# medium square in center
sink_sq = rectangle_sink(N, 20, 30, 20, 30)  # 10x10

# same area (100) but higher perimeter: 5x20 bar
sink_bar = rectangle_sink(N, 22, 27, 15, 35)  # 5x20

# same square near top and near bottom
sink_sq_top = rectangle_sink(N, 2, 12, 20, 30)
sink_sq_bot = rectangle_sink(N, 37, 47, 20, 30)  # keep < N-1

# two objects, total area 100 (two 5x10 rectangles)
s1 = rectangle_sink(N, 10, 15, 5, 15)   # 5x10
s2 = rectangle_sink(N, 30, 35, 35, 45)  # 5x10
sink_two = combine_sinks(s1, s2)

In [ ]:
tol = 1e-5
max_iter = 10000

cases = {
    "no sink": None,
    "square center (10x10)": sink_sq,
    "bar (5x20)": sink_bar,
    "square top (10x10)": sink_sq_top,
    "square bottom (10x10)": sink_sq_bot,
    "two rectangles (2x 5x10)": sink_two,
}

results = {}
for name, s in cases.items():
    w_opt, k_opt, _, _ = omega_opt_for_case(N, sink=s, tol=tol, max_iter=max_iter)
    results[name] = (w_opt, k_opt)
    print(f"{name:28s} ω_opt={w_opt:.2f}, k_min={k_opt}")

## L)

In [ ]:
N = 50
w = 1.92
tol = 1e-5

ins = rectangle_insulator(N, 10, 20, 10, 40)  

c0, d0, k0, _ = sor(N, w, tol=tol, sink=None, insulator=empty_insulator(N))
c1, d1, k1, _ = sor(N, w, tol=tol, sink=None, insulator=ins)


In [ ]:
plt.figure()
plt.imshow(c0, aspect='auto')
plt.colorbar()
plt.title(f"Concentration without insulator (k={k0})")


In [ ]:
plt.figure()
plt.imshow(c1, aspect='auto')
plt.colorbar()
plt.title(f"Concentration with insulator (k={k1})")


In [ ]:
def animate_field_html(history, *, sink=None, insulator=None,
                       interval_ms=50, fps=20, title_prefix="Iteration", show_colorbar=True):
    """
    Create an HTML (JS) animation of the concentration field over iterations.

    Parameters
    ----------
    history : list[np.ndarray]
        List of (N,N) arrays, one per stored iteration.
    sink, insulator : np.ndarray[bool] or None
        Optional masks to overlay.
    interval_ms : int
        Delay between frames in milliseconds.
    fps : int
        Used only for JS HTML export.
    title_prefix : str
        Prefix for title (iteration counter appended).
    show_colorbar : bool
        Whether to show a colorbar.

    Returns
    -------
    html : IPython.display.HTML
        Display this in a notebook cell. You can also save html.data to a file.
    """
    if len(history) == 0:
        raise ValueError("history is empty. Run sor(..., ret_hist=True) first.")



    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(history[0], aspect="auto")
    ax.set_xlabel("index j")
    ax.set_ylabel("index i")
    ttl = ax.set_title(f"{title_prefix} 0")


    if show_colorbar:
        cb = fig.colorbar(im, ax=ax, fraction=0.046)
        cb.set_label("c")

    def update(frame_idx):
        im.set_data(history[frame_idx])
        ttl.set_text(f"{title_prefix} {frame_idx}")
        return (im, ttl)

    anim = FuncAnimation(fig, update, frames=len(history), interval=interval_ms, blit=False)

    plt.close(fig)

    return HTML(anim.to_jshtml(fps=fps))

In [ ]:

c, deltas, hist, k, conv = sor(N, omega=w, tol=tol,
                                   sink=None, insulator=ins,
                                   ret_hist=True)


In [ ]:
animate_field_html(hist, insulator=ins, interval_ms=10000)


In [ ]:

html_obj = animate_field_html(hist, insulator=ins)

f_path = "../report/images/insulator_animation.html"
with open(f_path, "w") as f:
    f.write(html_obj.data)